# ⚙️ Notebook 2: Feature Engineering
## Climate Change and Vegetation Dynamics Analysis
---
**Objective:** Transform raw climate-vegetation data into a rich feature set
that maximises machine learning model performance.

**Key concepts covered:**
- Temporal lag features (vegetation memory)
- Rolling window statistics (medium-term climate stress)
- Cyclical encoding (seasonal patterns)
- Climate anomaly detection
- Vapour Pressure Deficit (VPD) — a key biophysical variable
- Feature selection and importance ranking

## 1. Setup

In [ ]:
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, "../src")
sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.dpi": 130, "figure.facecolor": "white"})

print("✅ Setup complete")

## 2. Load Clean Data

In [ ]:
from data_loader import DataLoader
from preprocessing import Preprocessor

# Load raw data
loader = DataLoader(random_seed=42, n_locations=50)
df_raw = loader.load_all_data()
df_raw["date"] = pd.to_datetime(df_raw["date"])

# Clean it
prep = Preprocessor()
df_clean = prep.clean_data(df_raw)

print(f"Raw shape:   {df_raw.shape}")
print(f"Clean shape: {df_clean.shape}")
print(f"Columns: {list(df_clean.columns)}")

## 3. Cyclical Month Encoding

**The problem with raw month numbers:**
Month 12 (December) and Month 1 (January) are adjacent, but if we pass
raw numbers (1–12), the model treats them as 11 units apart.

**The solution — sine/cosine encoding:**
We encode month as two components on a unit circle. This preserves
the cyclic nature: December → January is a tiny step, not a big jump.

```
month_sin = sin(2π × month / 12)
month_cos = cos(2π × month / 12)
```

In [ ]:
months = np.arange(1, 13)
month_sin = np.sin(2 * np.pi * months / 12)
month_cos = np.cos(2 * np.pi * months / 12)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Raw encoding (BAD)
axes[0].plot(months, months, "o-", color="#E74C3C", lw=2)
axes[0].set_xticks(months)
axes[0].set_title("Raw Month Encoding (WRONG)\nDec–Jan gap = 11 units", fontweight="bold")
axes[0].set_xlabel("Month"); axes[0].set_ylabel("Encoded Value")

# Cyclical encoding (GOOD)
theta = np.linspace(0, 2*np.pi, 200)
axes[1].plot(np.cos(theta), np.sin(theta), "gray", alpha=0.3)
sc = axes[1].scatter(month_cos, month_sin, c=months, cmap="RdYlGn", s=120, zorder=3)
for m, x, y in zip(months, month_cos, month_sin):
    axes[1].annotate(["","Jan","Feb","Mar","Apr","May","Jun",
                       "Jul","Aug","Sep","Oct","Nov","Dec"][m],
                     (x, y), fontsize=8, ha="center", va="bottom")
plt.colorbar(sc, ax=axes[1], label="Month")
axes[1].set_title("Cyclical Encoding (CORRECT)\nDec and Jan are adjacent", fontweight="bold")
axes[1].set_aspect("equal"); axes[1].set_xlabel("cos"); axes[1].set_ylabel("sin")

plt.tight_layout()
plt.savefig("../images/fe_cyclical_encoding.png", dpi=130, bbox_inches="tight")
plt.show()

## 4. Lag Features — Vegetation Memory

Plants don't respond to climate conditions instantaneously. A rainfall
event this month may not affect NDVI until next month (root uptake,
stomatal response). This delay is called **vegetation memory**.

We create lag features: NDVI and precipitation values from 1, 2, 3 months ago.

In [ ]:
# Demonstrate lag features on one location
loc0 = df_clean[df_clean["location_id"] == 0].sort_values("date").copy()

for lag in [1, 2, 3]:
    loc0[f"ndvi_lag_{lag}"] = loc0["ndvi"].shift(lag)
    loc0[f"precip_lag_{lag}"] = loc0["precipitation"].shift(lag)

print("Lag feature example (first 8 rows):")
print(loc0[["date", "ndvi", "ndvi_lag_1", "ndvi_lag_2", "ndvi_lag_3",
             "precipitation", "precip_lag_1"]].head(8).to_string(index=False))

# Visualise lag correlation
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Lag Correlation: NDVI(t) vs NDVI(t-lag)", fontweight="bold", fontsize=13)
for ax, lag in zip(axes, [1, 2, 3]):
    col = f"ndvi_lag_{lag}"
    valid = loc0.dropna(subset=[col])
    ax.scatter(valid[col], valid["ndvi"], alpha=0.4, s=10, color="#27AE60")
    r = valid[["ndvi", col]].corr().iloc[0, 1]
    ax.set_title(f"Lag = {lag} month (r = {r:.3f})")
    ax.set_xlabel(f"NDVI(t-{lag})")
    ax.set_ylabel("NDVI(t)")
plt.tight_layout()
plt.savefig("../images/fe_lag_correlation.png", dpi=130, bbox_inches="tight")
plt.show()

## 5. Rolling Statistics

In [ ]:
loc0["ndvi_roll_3"]  = loc0["ndvi"].rolling(3,  min_periods=1).mean()
loc0["ndvi_roll_6"]  = loc0["ndvi"].rolling(6,  min_periods=1).mean()
loc0["precip_sum_3"] = loc0["precipitation"].rolling(3, min_periods=1).sum()

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
fig.suptitle("Rolling Statistics vs Raw NDVI", fontsize=13, fontweight="bold")

axes[0].plot(loc0["date"], loc0["ndvi"], alpha=0.5, color="#27AE60", lw=1.2, label="Monthly NDVI")
axes[0].plot(loc0["date"], loc0["ndvi_roll_3"], color="#1A5276", lw=2, label="3-mo rolling mean")
axes[0].plot(loc0["date"], loc0["ndvi_roll_6"], color="#E74C3C", lw=2, label="6-mo rolling mean", ls="--")
axes[0].set_ylabel("NDVI"); axes[0].legend(fontsize=9)
axes[0].set_title("NDVI — Rolling Mean Smoothing")

axes[1].bar(loc0["date"], loc0["precipitation"], alpha=0.4, color="#3498DB", label="Monthly precip")
axes[1].plot(loc0["date"], loc0["precip_sum_3"], color="#1A5276", lw=2, label="3-mo cumulative")
axes[1].set_ylabel("Precipitation (mm)"); axes[1].legend(fontsize=9)
axes[1].set_title("Precipitation — 3-Month Cumulative")

plt.tight_layout()
plt.savefig("../images/fe_rolling_stats.png", dpi=130, bbox_inches="tight")
plt.show()

## 6. Vapour Pressure Deficit (VPD)

**VPD** is the difference between the amount of moisture the air *can* hold
and the amount it *actually* holds. It's one of the most important drivers
of plant water stress.

High VPD → atmosphere pulls water out of leaves → stomata close → less photosynthesis → NDVI drops.

```
Saturation VP  = 0.6108 × exp(17.27 × T / (T + 237.3))   [kPa]
Actual VP      = Saturation VP × (RH / 100)
VPD            = Saturation VP × (1 − RH/100)
```

In [ ]:
T  = df_clean["temperature_mean"]
RH = df_clean["humidity"]
sat_vp = 0.6108 * np.exp(17.27 * T / (T + 237.3))
df_clean = df_clean.copy()
df_clean["vpd_kpa"] = sat_vp * (1 - RH / 100)

print("VPD statistics:")
print(df_clean["vpd_kpa"].describe().round(3))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(df_clean["vpd_kpa"].sample(3000, random_state=1),
                df_clean["ndvi"].sample(3000, random_state=1),
                alpha=0.2, s=8, color="#E74C3C")
axes[0].set_xlabel("VPD (kPa)"); axes[0].set_ylabel("NDVI")
axes[0].set_title("NDVI vs VPD\n(High VPD = plant stress = low NDVI)")

sns.kdeplot(data=df_clean, x="vpd_kpa", hue="region", fill=True, alpha=0.25, ax=axes[1],
            palette={"Sahel":"#E67E22","Savanna":"#F1C40F","Rainforest":"#27AE60",
                     "Semi-Arid":"#BDC3C7","Mediterranean":"#3498DB"})
axes[1].set_title("VPD Distribution by Region")
axes[1].set_xlabel("VPD (kPa)")
plt.tight_layout()
plt.savefig("../images/fe_vpd.png", dpi=130, bbox_inches="tight")
plt.show()

## 7. Run the Full Feature Engineering Pipeline

In [ ]:
# Use the Preprocessor class to run all steps in one call
prep = Preprocessor()
df_features = prep.engineer_features(df_clean)

print(f"\nBefore engineering: {df_clean.shape[1]} columns")
print(f"After engineering:  {df_features.shape[1]} columns")
print(f"New features: {df_features.shape[1] - df_clean.shape[1]}")
print(f"\nNew columns added:")
new_cols = [c for c in df_features.columns if c not in df_clean.columns]
for c in new_cols:
    print(f"  + {c}")

## 8. Feature Importance via Mutual Information

In [ ]:
from sklearn.feature_selection import mutual_info_regression

# Prepare X and y
X_raw, y = prep.select_features(df_features)
X_arr = X_raw.fillna(0).values

# Mutual information measures non-linear statistical dependence
# (unlike Pearson correlation, it captures nonlinear relationships)
mi_scores = mutual_info_regression(X_arr, y.values, random_state=42)

mi_df = pd.DataFrame({
    "feature": prep.feature_columns,
    "mutual_info": mi_scores
}).sort_values("mutual_info", ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
n = min(20, len(mi_df))
top = mi_df.head(n)
colors = plt.cm.Greens(np.linspace(0.4, 0.9, n))[::-1]
ax.barh(top["feature"][::-1], top["mutual_info"][::-1], color=colors, edgecolor="white")
ax.set_xlabel("Mutual Information Score")
ax.set_title(f"Top {n} Features — Mutual Information with NDVI", fontweight="bold", fontsize=13)
plt.tight_layout()
plt.savefig("../images/fe_mutual_info.png", dpi=130, bbox_inches="tight")
plt.show()

print("\nTop 10 most informative features:")
print(mi_df.head(10).to_string(index=False))

## 9. Standardisation

In [ ]:
X_train, X_test, y_train, y_test = prep.split_data(X_raw, y, method="temporal")
X_train_s, X_test_s = prep.scale_features(X_train, X_test)

print(f"Training samples : {X_train_s.shape[0]:,}")
print(f"Test samples     : {X_test_s.shape[0]:,}")
print(f"Features         : {X_train_s.shape[1]}")
print(f"\nPre-scaling  — mean: {X_train.values.mean():+.4f}, std: {X_train.values.std():.4f}")
print(f"Post-scaling — mean: {X_train_s.mean():+.6f}, std: {X_train_s.std():.4f}")

## 10. Summary

### Features Created

| Category | Features |
|---|---|
| Cyclical time | `month_sin`, `month_cos` |
| NDVI lags | `ndvi_lag_1/2/3` |
| Precip lags | `precip_lag_1/2/3` |
| Temperature lag | `temp_lag_1` |
| Rolling means | `ndvi_roll_mean_3/6`, `temp_roll_mean_3` |
| Rolling sum | `precip_roll_sum_3/6` |
| Biophysical | `vpd_kpa`, `aridity_index` |
| Interactions | `temp_x_precip` |
| Anomalies | `temp_anomaly`, `precip_anomaly` |
| Region | `region_Savanna`, `region_Rainforest`, ... |

### 🔜 Next Step
Proceed to **`03_Model_Training.ipynb`** to train and compare ML models.